In [12]:
from bs4 import BeautifulSoup
import requests
import os
def main():
    url = "https://news.google.com/news/headlines?ned=in&hl=en-IN&gl=IN"
    data = requests.get(url)
    soup = BeautifulSoup(data.content, "html.parser")
    filename = "d:/clustering/data/scrapped.txt"
    links = soup.find_all("a")
    with open(filename, 'w') as f:
        for link in links:
            text = link.text
            
            headline_length = len(text.split())
            if headline_length >= 3:
                f.write(text)
                f.write('\n')
if __name__ == '__main__':
    main()

In [14]:
import re
from string import punctuation
import os

def main():
    #for jupyter
    f = open("d:/clustering/data/scrapped.txt", 'rt')#rt means read in text mode
    text_file = f.read().split('\n')
    text_lower = [text.lower() for text in text_file]
    text_letters = [''.join(c for c in s if c not in punctuation) for s in text_lower]
    text_final = [re.sub(r'[^A-Za-z\d]+', ' ', x) for x in text_letters]
    with open("d:/clustering/data/cleaned.txt",'w') as fw:
        for text in text_final:
            fw.write(text)
            fw.write('\n')
if __name__ == '__main__':
    main()


In [19]:
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
from sklearn.manifold import MDS

import pandas as pd
from sklearn.cluster import KMeans
import os

def tokenize_and_stem(text_file):
    # declaring stemmer and stopwords language
    stemmer = SnowballStemmer("english")
    stop_words = set(stopwords.words('english'))
    words = word_tokenize(text_file)
    filtered = [w for w in words if w not in stop_words]
    stems = [stemmer.stem(t) for t in filtered]
    return stems


def main():

    data = pd.read_csv("d:/clustering/data/cleaned.txt",names=['text'])

    # text data in dataframe and removing stops words
    stop_words = set(stopwords.words('english'))
    data['text'] = data['text'].apply(lambda x: ' '.join([word for word in x.split() if word not in stop_words]))

    # Using TFIDF vectorizer to convert words 
    #to Vector Space
    tfidf_vectorizer = TfidfVectorizer(max_features=200000,use_idf=True,
    stop_words='english',tokenizer=tokenize_and_stem)

    # Fit the vectorizer to text data
    tfidf_matrix = tfidf_vectorizer.fit_transform(data['text'])
    terms = tfidf_vectorizer.get_feature_names_out()
    # print(terms)

    # Kmeans++
    km = KMeans(n_clusters=7, init='k-means++', max_iter=300, n_init=1,random_state=34)
    km.fit(tfidf_matrix)
    labels = km.labels_
    clusters = labels.tolist()
	#print(cluster)
    # Calculating the distance measure derived from cosine 
    #similarity
    distance = 1 - cosine_similarity(tfidf_matrix)

    # Dimensionality reduction using Multidimensional 
    #scaling (MDS)
    mds = MDS(n_components=2, dissimilarity="precomputed", random_state=1)
    pos = mds.fit_transform(distance)
    xs, ys = pos[:, 0], pos[:, 1]

    # Saving cluster visualization after mutidimensional scaling
    
    # Creating dataframe containing reduced dimensions, 
    #identified labels and text data for plotting 
    #KMeans output
    df = pd.DataFrame(dict(label=clusters, data=data['text'], x=xs, y=ys))
    
    df.to_csv("d:/clustering/results/kmeans_clustered_DF.txt", sep=',')
    
if __name__ == '__main__':
    main()


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:411: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['afterward', 'alon', 'alreadi', 'alway', 'anoth', 'anyon', 'anyth', 'anywher', 'becam', 'becom', 'besid', 'cri', 'describ', 'els', 'elsewher', 'empti', 'everi', 'everyon', 'everyth', 'everywher', 'fifti', 'forti', 'henc', 'hereaft', 'herebi', 'howev', 'hundr', 'inde', 'mani', 'meanwhil', 'moreov', 'nobodi', 'noon', 'noth', 'nowher', 'otherwis', 'perhap', 'pleas', 'sever', 'sinc', 'sincer', 'sixti', 'someon', 'someth', 'sometim', 'somewher', 'thenc', 'thereaft', 'therebi', 'therefor', 'togeth', 'twelv', 'twenti', 'whatev', 'whenc', 'whenev', 'wherea', 'whereaft', 'wherebi', 'wherev'] not in stop_words.